# Notebook 04a — Deep Learning Architecture Search

**Question:** Which DL architectures work best on tabular startup data?

We test 4 model families on the 3-class and binary framings:
1. **MLP** — 3 architecture variants (wide, deep, original), pick the best
2. **TabNet** — attention-based feature selection (Google Research, 2019)
3. **TabTransformer** — self-attention over feature embeddings (2020)
4. **FT-Transformer** — Feature Tokenizer + Transformer (Gorishniy et al., 2021)

All models use weighted CrossEntropyLoss (DL equivalent of `class_weight='balanced'`),
Adam optimizer, ReduceLROnPlateau scheduler, and early stopping on val macro F1.

**Inputs:** `artifacts/` from nb02
**Outputs:** `results/dl_architectures.csv`

> **Runtime note:** Designed for Google Colab. Use GPU runtime for faster training.


In [ ]:
# ── Install dependencies ──
# Uncomment these lines on Colab:
# !pip install pytorch-tabnet -q

import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, time, os, joblib
import matplotlib.pyplot as plt, seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_class_weight

plt.rcParams['figure.figsize'] = (12, 5)
sns.set_style("whitegrid")

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

os.makedirs('results', exist_ok=True)
os.makedirs('models', exist_ok=True)
print("Imports OK")


## 0. Load Artifacts & Prepare Data

In [ ]:
# Load preprocessed data from nb02
X_train_3c = joblib.load('artifacts/X_train_3c.pkl').astype(np.float32)
X_val_3c   = joblib.load('artifacts/X_val_3c.pkl').astype(np.float32)
X_test_3c  = joblib.load('artifacts/X_test_3c.pkl').astype(np.float32)
y_train_3c = joblib.load('artifacts/y_train_3c.pkl')
y_val_3c   = joblib.load('artifacts/y_val_3c.pkl')
y_test_3c  = joblib.load('artifacts/y_test_3c.pkl')

X_train_bin = joblib.load('artifacts/X_train_bin.pkl').astype(np.float32)
X_val_bin   = joblib.load('artifacts/X_val_bin.pkl').astype(np.float32)
X_test_bin  = joblib.load('artifacts/X_test_bin.pkl').astype(np.float32)
y_train_bin = joblib.load('artifacts/y_train_bin.pkl')
y_val_bin   = joblib.load('artifacts/y_val_bin.pkl')
y_test_bin  = joblib.load('artifacts/y_test_bin.pkl')

le_3class     = joblib.load('artifacts/label_encoder_3class.pkl')
feature_names = joblib.load('artifacts/feature_names.pkl')

N_FEATURES = X_train_3c.shape[1]
N_CLASSES_3 = len(le_3class.classes_)
N_CLASSES_BIN = 2

print(f'Features: {N_FEATURES}')
print(f'3-class: train={X_train_3c.shape[0]}, val={X_val_3c.shape[0]}, test={X_test_3c.shape[0]}')
print(f'Binary:  train={X_train_bin.shape[0]}, val={X_val_bin.shape[0]}, test={X_test_bin.shape[0]}')
print(f'3-class labels: {le_3class.classes_.tolist()}')


In [ ]:
# ── Compute class weights for weighted loss ──
weights_3c = compute_class_weight('balanced', classes=np.unique(y_train_3c), y=y_train_3c)
weights_3c = torch.FloatTensor(weights_3c).to(DEVICE)
print(f'3-class weights: {weights_3c.cpu().numpy().round(3)}')

weights_bin = compute_class_weight('balanced', classes=np.unique(y_train_bin), y=y_train_bin)
weights_bin = torch.FloatTensor(weights_bin).to(DEVICE)
print(f'Binary weights:  {weights_bin.cpu().numpy().round(3)}')

# ── Create DataLoaders ──
BATCH_SIZE = 256

def make_loaders(X_tr, y_tr, X_v, y_v, batch_size=BATCH_SIZE):
    train_ds = TensorDataset(torch.FloatTensor(X_tr), torch.LongTensor(y_tr))
    val_ds   = TensorDataset(torch.FloatTensor(X_v), torch.LongTensor(y_v))
    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_dl   = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
    return train_dl, val_dl

train_dl_3c, val_dl_3c = make_loaders(X_train_3c, y_train_3c, X_val_3c, y_val_3c)
train_dl_bin, val_dl_bin = make_loaders(X_train_bin, y_train_bin, X_val_bin, y_val_bin)


## 1. Training Infrastructure

A reusable training loop shared by all DL models. Handles:
- Weighted CrossEntropyLoss
- Adam optimizer + ReduceLROnPlateau
- Early stopping on val macro F1
- Learning curve tracking


In [ ]:
def train_model(model, train_dl, val_dl, class_weights, n_classes,
                epochs=100, lr=1e-3, patience=10, min_delta=0.001):
    """
    Train a PyTorch model with early stopping on val macro F1.
    Returns: trained model, history dict
    """
    model = model.to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=5, min_lr=1e-6)

    history = {'train_loss': [], 'val_loss': [], 'val_f1': [], 'lr': []}
    best_f1 = 0
    best_state = None
    wait = 0

    for epoch in range(epochs):
        # ── Train ──
        model.train()
        train_losses = []
        for X_batch, y_batch in train_dl:
            X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
            optimizer.zero_grad()
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        # ── Validate ──
        model.eval()
        val_losses, all_preds, all_true = [], [], []
        with torch.no_grad():
            for X_batch, y_batch in val_dl:
                X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
                logits = model(X_batch)
                loss = criterion(logits, y_batch)
                val_losses.append(loss.item())
                preds = logits.argmax(dim=1).cpu().numpy()
                all_preds.extend(preds)
                all_true.extend(y_batch.cpu().numpy())

        val_f1 = f1_score(all_true, all_preds, average='macro')
        current_lr = optimizer.param_groups[0]['lr']
        scheduler.step(val_f1)

        history['train_loss'].append(np.mean(train_losses))
        history['val_loss'].append(np.mean(val_losses))
        history['val_f1'].append(val_f1)
        history['lr'].append(current_lr)

        # ── Early stopping ──
        if val_f1 > best_f1 + min_delta:
            best_f1 = val_f1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1

        if (epoch + 1) % 10 == 0 or wait == 0:
            print(f"  Epoch {epoch+1:3d} | loss={np.mean(train_losses):.4f} | "
                  f"val_loss={np.mean(val_losses):.4f} | val_F1={val_f1:.4f} | "
                  f"lr={current_lr:.1e} | {'*' if wait==0 else ''}")

        if wait >= patience:
            print(f"  Early stopping at epoch {epoch+1} (best F1={best_f1:.4f})")
            break

    # Restore best weights
    if best_state is not None:
        model.load_state_dict(best_state)
    model = model.to(DEVICE)

    return model, history, best_f1


def plot_history(history, title=""):
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    axes[0].plot(history['train_loss'], label='Train Loss', color='#378ADD')
    axes[0].plot(history['val_loss'], label='Val Loss', color='#D85A30')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
    axes[0].set_title(f'{title} — Loss Curves')
    axes[0].legend()

    axes[1].plot(history['val_f1'], label='Val Macro F1', color='#1D9E75')
    best_ep = np.argmax(history['val_f1'])
    axes[1].axvline(x=best_ep, color='gray', linestyle='--', alpha=0.5)
    axes[1].annotate(f'Best: {history["val_f1"][best_ep]:.4f}',
                     xy=(best_ep, history['val_f1'][best_ep]),
                     xytext=(best_ep+5, history['val_f1'][best_ep]-0.02),
                     fontsize=10, color='#1D9E75')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Macro F1')
    axes[1].set_title(f'{title} — Validation F1')
    axes[1].legend()

    plt.tight_layout()
    plt.show()


In [ ]:
# Experiment tracker (same format as ML notebooks)
SCOREBOARD = []

def log_exp(model, strategy, val_f1, n_train, notes=""):
    SCOREBOARD.append(dict(
        phase="dl_arch", model=model, strategy=strategy,
        n_features=N_FEATURES,
        cv_f1_mean=round(val_f1, 4), cv_f1_std=0,
        n_train=n_train, notes=notes
    ))
    print(f"  >> {model:25s} | {strategy:8s} | val F1={val_f1:.4f}")


## 2. MLP — Architecture Search

We test 3 MLP variants to find the best architecture before tuning:
- **Wide:** 512 → 256 → output (fewer layers, more capacity per layer)
- **Deep:** 128 → 128 → 128 → 64 → output (more layers, hierarchical)
- **Medium:** 256 → 128 → 64 → output (balanced, original nb04 architecture)

All share: BatchNorm, Dropout(0.3), ReLU activation.


In [ ]:
class MLP(nn.Module):
    """Flexible MLP with BatchNorm and Dropout."""
    def __init__(self, n_input, n_output, hidden_dims, dropout=0.3):
        super().__init__()
        layers = []
        prev_dim = n_input
        for h in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, h),
                nn.BatchNorm1d(h),
                nn.ReLU(),
                nn.Dropout(dropout),
            ])
            prev_dim = h
        layers.append(nn.Linear(prev_dim, n_output))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

# Architecture variants
MLP_VARIANTS = {
    "MLP-Wide":   [512, 256],
    "MLP-Deep":   [128, 128, 128, 64],
    "MLP-Medium": [256, 128, 64],
}

print("MLP architectures defined:")
for name, dims in MLP_VARIANTS.items():
    total_params = sum(
        (N_FEATURES if i==0 else dims[i-1]) * d + d  # weights + bias
        for i, d in enumerate(dims)
    ) + dims[-1] * N_CLASSES_3 + N_CLASSES_3
    print(f"  {name}: {' → '.join(map(str, [N_FEATURES] + dims + ['output']))} ({total_params:,} params)")


In [ ]:
# ── MLP Architecture Search on 3-class ──
print("=" * 60)
print("MLP Architecture Search — 3-class")
print("=" * 60)

mlp_results = {}
for name, hidden_dims in MLP_VARIANTS.items():
    print(f"\n--- {name}: {hidden_dims} ---")
    model = MLP(N_FEATURES, N_CLASSES_3, hidden_dims, dropout=0.3)
    model, hist, best_f1 = train_model(
        model, train_dl_3c, val_dl_3c, weights_3c, N_CLASSES_3,
        epochs=100, lr=1e-3, patience=15)
    mlp_results[name] = {'f1': best_f1, 'hist': hist, 'dims': hidden_dims}
    log_exp(name, "3class", best_f1, len(y_train_3c), f"arch_search")
    plot_history(hist, name)

best_mlp_name = max(mlp_results, key=lambda k: mlp_results[k]['f1'])
best_mlp_dims = mlp_results[best_mlp_name]['dims']
print(f"\nBest MLP: {best_mlp_name} (F1={mlp_results[best_mlp_name]['f1']:.4f})")
print(f"Using {best_mlp_name} for all subsequent MLP experiments.")


In [ ]:
# ── Best MLP on binary ──
print("\n--- Best MLP on binary ---")
model = MLP(N_FEATURES, N_CLASSES_BIN, best_mlp_dims, dropout=0.3)
model, hist, best_f1 = train_model(
    model, train_dl_bin, val_dl_bin, weights_bin, N_CLASSES_BIN,
    epochs=100, lr=1e-3, patience=15)
log_exp(best_mlp_name, "binary", best_f1, len(y_train_bin))
plot_history(hist, f"{best_mlp_name} (binary)")


## 3. TabNet

TabNet (Arik & Pfister, 2019) uses **sequential attention-based feature selection**.
At each decision step, it applies a learnable sparsemax mask to select which features
to attend to. This is fundamentally different from both MLP (all features equally)
and trees (one feature per split).

We use `pytorch-tabnet` which provides a sklearn-like API.


In [ ]:
from pytorch_tabnet.tab_model import TabNetClassifier

# ── TabNet on 3-class ──
print("=" * 60)
print("TabNet — 3-class")
print("=" * 60)

tabnet_3c = TabNetClassifier(
    n_d=32, n_a=32,           # decision & attention dimensions
    n_steps=5,                 # number of sequential attention steps
    gamma=1.5,                 # coefficient for feature reuse
    lambda_sparse=1e-3,        # sparsity regularization
    optimizer_fn=torch.optim.Adam,
    optimizer_params=dict(lr=2e-2, weight_decay=1e-5),
    scheduler_fn=torch.optim.lr_scheduler.StepLR,
    scheduler_params=dict(step_size=10, gamma=0.9),
    mask_type='sparsemax',
    verbose=10,
    seed=SEED,
    device_name=DEVICE.type,
)

tabnet_3c.fit(
    X_train_3c, y_train_3c,
    eval_set=[(X_val_3c, y_val_3c)],
    eval_metric=['balanced_accuracy'],
    max_epochs=100,
    patience=15,
    batch_size=256,
    virtual_batch_size=128,
    weights=dict(enumerate(weights_3c.cpu().numpy())),
)

# Evaluate
y_pred = tabnet_3c.predict(X_val_3c)
f1_tabnet_3c = f1_score(y_val_3c, y_pred, average='macro')
log_exp("TabNet", "3class", f1_tabnet_3c, len(y_train_3c))

# Plot TabNet's built-in loss history
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(tabnet_3c.history['loss'], label='Train Loss')
if 'val_0_balanced_accuracy' in tabnet_3c.history:
    ax2 = ax.twinx()
    ax2.plot(tabnet_3c.history['val_0_balanced_accuracy'], color='#1D9E75', label='Val Balanced Acc')
    ax2.set_ylabel('Balanced Accuracy')
    ax2.legend(loc='upper left')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.set_title('TabNet — 3-class Training History')
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()


In [ ]:
# ── TabNet on binary ──
print("\n--- TabNet on binary ---")

tabnet_bin = TabNetClassifier(
    n_d=32, n_a=32, n_steps=5, gamma=1.5, lambda_sparse=1e-3,
    optimizer_fn=torch.optim.Adam,
    optimizer_params=dict(lr=2e-2, weight_decay=1e-5),
    scheduler_fn=torch.optim.lr_scheduler.StepLR,
    scheduler_params=dict(step_size=10, gamma=0.9),
    mask_type='sparsemax', verbose=10, seed=SEED,
    device_name=DEVICE.type,
)

tabnet_bin.fit(
    X_train_bin, y_train_bin,
    eval_set=[(X_val_bin, y_val_bin)],
    eval_metric=['balanced_accuracy'],
    max_epochs=100, patience=15,
    batch_size=256, virtual_batch_size=128,
    weights=dict(enumerate(weights_bin.cpu().numpy())),
)

y_pred = tabnet_bin.predict(X_val_bin)
f1_tabnet_bin = f1_score(y_val_bin, y_pred, average='macro')
log_exp("TabNet", "binary", f1_tabnet_bin, len(y_train_bin))


In [ ]:
# ── TabNet Feature Importance (attention masks) ──
print("\nTabNet Feature Importance (3-class):")
feat_imp = tabnet_3c.feature_importances_
imp_df = pd.DataFrame({
    'feature': feature_names,
    'importance': feat_imp
}).sort_values('importance', ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
top_n = min(15, len(imp_df))
imp_top = imp_df.head(top_n).sort_values('importance')
ax.barh(imp_top['feature'], imp_top['importance'], color='#534AB7', edgecolor='white')
ax.set_xlabel('TabNet Attention-based Importance')
ax.set_title('TabNet Feature Importance — 3-class (Top 15)')
plt.tight_layout()
plt.savefig('results/tabnet_feature_importance.png', bbox_inches='tight')
plt.show()
print(imp_df.head(10).to_string(index=False))


## 4. TabTransformer

TabTransformer (Huang et al., 2020) embeds each feature into a high-dimensional space,
then applies multi-head self-attention so features can interact. This lets the model
learn complex feature combinations that a simple MLP misses.

We test 2 configurations:
- **Small:** d_model=64, 2 layers, 4 heads
- **Large:** d_model=128, 3 layers, 8 heads


In [ ]:
class TabTransformer(nn.Module):
    """
    Transformer for tabular data.
    Each feature is projected to d_model dimensions, then processed
    by self-attention layers, and finally classified via an MLP head.
    """
    def __init__(self, n_features, n_classes, d_model=64, n_heads=4,
                 n_layers=2, dim_ff=128, dropout=0.2):
        super().__init__()
        self.n_features = n_features
        self.d_model = d_model

        # Per-feature linear projection to d_model
        self.feature_embeddings = nn.ModuleList([
            nn.Linear(1, d_model) for _ in range(n_features)
        ])

        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=dim_ff,
            dropout=dropout, batch_first=True, activation='gelu'
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)

        # Layer norm
        self.ln = nn.LayerNorm(d_model)

        # Classification head
        self.head = nn.Sequential(
            nn.Linear(d_model * n_features, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, n_classes),
        )

    def forward(self, x):
        # x: (batch, n_features)
        # Embed each feature separately
        embeddings = []
        for i in range(self.n_features):
            emb = self.feature_embeddings[i](x[:, i:i+1])  # (batch, d_model)
            embeddings.append(emb)

        # Stack into sequence: (batch, n_features, d_model)
        tokens = torch.stack(embeddings, dim=1)

        # Self-attention
        tokens = self.transformer(tokens)
        tokens = self.ln(tokens)

        # Flatten and classify
        flat = tokens.reshape(tokens.size(0), -1)
        return self.head(flat)


TABTRANS_CONFIGS = {
    "TabTrans-Small": dict(d_model=64, n_heads=4, n_layers=2, dim_ff=128),
    "TabTrans-Large": dict(d_model=128, n_heads=8, n_layers=3, dim_ff=256),
}

for name, cfg in TABTRANS_CONFIGS.items():
    model = TabTransformer(N_FEATURES, N_CLASSES_3, **cfg)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"{name}: {n_params:,} parameters")


In [ ]:
# ── TabTransformer Architecture Search on 3-class ──
print("=" * 60)
print("TabTransformer — 3-class")
print("=" * 60)

tabtrans_results = {}
for name, cfg in TABTRANS_CONFIGS.items():
    print(f"\n--- {name}: {cfg} ---")
    model = TabTransformer(N_FEATURES, N_CLASSES_3, dropout=0.2, **cfg)
    model, hist, best_f1 = train_model(
        model, train_dl_3c, val_dl_3c, weights_3c, N_CLASSES_3,
        epochs=80, lr=1e-3, patience=12)
    tabtrans_results[name] = {'f1': best_f1, 'hist': hist, 'cfg': cfg}
    log_exp(name, "3class", best_f1, len(y_train_3c))
    plot_history(hist, name)

best_tt_name = max(tabtrans_results, key=lambda k: tabtrans_results[k]['f1'])
best_tt_cfg = tabtrans_results[best_tt_name]['cfg']
print(f"\nBest TabTransformer: {best_tt_name} (F1={tabtrans_results[best_tt_name]['f1']:.4f})")


In [ ]:
# ── Best TabTransformer on binary ──
print("\n--- Best TabTransformer on binary ---")
model = TabTransformer(N_FEATURES, N_CLASSES_BIN, dropout=0.2, **best_tt_cfg)
model, hist, best_f1 = train_model(
    model, train_dl_bin, val_dl_bin, weights_bin, N_CLASSES_BIN,
    epochs=80, lr=1e-3, patience=12)
log_exp(best_tt_name, "binary", best_f1, len(y_train_bin))
plot_history(hist, f"{best_tt_name} (binary)")


## 5. FT-Transformer

FT-Transformer (Gorishniy et al., 2021) improves on TabTransformer with two innovations:
1. **Feature Tokenizer:** Each feature gets a learnable embedding + bias (not just a linear projection)
2. **[CLS] Token:** A special classification token aggregates information across all features
   (like BERT), instead of flattening all feature embeddings

This was shown to be the strongest tabular transformer architecture in the original paper.


In [ ]:
class FTTransformer(nn.Module):
    """
    Feature Tokenizer + Transformer (Gorishniy et al., 2021).
    Each feature gets a learnable embedding. A [CLS] token aggregates
    information via self-attention for classification.
    """
    def __init__(self, n_features, n_classes, d_model=64, n_heads=4,
                 n_layers=2, dim_ff=128, dropout=0.2):
        super().__init__()
        self.n_features = n_features
        self.d_model = d_model

        # Feature Tokenizer: per-feature embedding + learnable bias
        self.feature_weights = nn.Parameter(torch.randn(n_features, d_model))
        self.feature_biases = nn.Parameter(torch.zeros(n_features, d_model))
        nn.init.xavier_uniform_(self.feature_weights)

        # Learnable [CLS] token
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))
        nn.init.normal_(self.cls_token, std=0.02)

        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=dim_ff,
            dropout=dropout, batch_first=True, activation='gelu'
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)

        # Layer norms
        self.ln_input = nn.LayerNorm(d_model)
        self.ln_output = nn.LayerNorm(d_model)

        # Classification head (only from [CLS] token)
        self.head = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, n_classes),
        )

    def forward(self, x):
        batch_size = x.size(0)

        # Feature Tokenizer: each feature → d_model embedding
        # x[:, i] * weight[i] + bias[i] for each feature
        # x: (batch, n_feat) → (batch, n_feat, 1) * (n_feat, d_model) + (n_feat, d_model)
        tokens = x.unsqueeze(-1) * self.feature_weights.unsqueeze(0) + self.feature_biases.unsqueeze(0)
        # tokens: (batch, n_features, d_model)

        tokens = self.ln_input(tokens)

        # Prepend [CLS] token
        cls = self.cls_token.expand(batch_size, -1, -1)  # (batch, 1, d_model)
        tokens = torch.cat([cls, tokens], dim=1)  # (batch, n_features+1, d_model)

        # Self-attention
        tokens = self.transformer(tokens)

        # Extract [CLS] token output (first position)
        cls_out = tokens[:, 0]  # (batch, d_model)
        cls_out = self.ln_output(cls_out)

        return self.head(cls_out)


model = FTTransformer(N_FEATURES, N_CLASSES_3)
n_params = sum(p.numel() for p in model.parameters())
print(f"FT-Transformer: {n_params:,} parameters")


In [ ]:
# ── FT-Transformer on 3-class ──
print("=" * 60)
print("FT-Transformer — 3-class")
print("=" * 60)

ft_model = FTTransformer(N_FEATURES, N_CLASSES_3, d_model=64, n_heads=4,
                          n_layers=2, dim_ff=128, dropout=0.2)
ft_model, ft_hist, ft_f1_3c = train_model(
    ft_model, train_dl_3c, val_dl_3c, weights_3c, N_CLASSES_3,
    epochs=80, lr=1e-3, patience=12)
log_exp("FT-Transformer", "3class", ft_f1_3c, len(y_train_3c))
plot_history(ft_hist, "FT-Transformer (3-class)")

# ── FT-Transformer on binary ──
print("\n--- FT-Transformer on binary ---")
ft_model_bin = FTTransformer(N_FEATURES, N_CLASSES_BIN, d_model=64, n_heads=4,
                              n_layers=2, dim_ff=128, dropout=0.2)
ft_model_bin, ft_hist_bin, ft_f1_bin = train_model(
    ft_model_bin, train_dl_bin, val_dl_bin, weights_bin, N_CLASSES_BIN,
    epochs=80, lr=1e-3, patience=12)
log_exp("FT-Transformer", "binary", ft_f1_bin, len(y_train_bin))
plot_history(ft_hist_bin, "FT-Transformer (binary)")


## 6. DL Architecture Comparison

In [ ]:
sb = pd.DataFrame(SCOREBOARD)

# Summary table
print("=" * 60)
print("DL Architecture Comparison")
print("=" * 60)
print(sb.sort_values('cv_f1_mean', ascending=False).to_string(index=False))

# Heatmap
pivot = sb[~sb['model'].str.contains('MLP-') | (sb['model'] == best_mlp_name)]
# Keep only the best variant of each architecture
best_per_arch = {}
for _, r in sb.iterrows():
    arch = r['model'].split('-')[0] if 'MLP' not in r['model'] else 'MLP'
    if 'MLP-' in r['model']:
        arch = 'MLP'
        # Only keep the best MLP
        if r['model'] != best_mlp_name:
            continue
    key = (arch, r['strategy'])
    if key not in best_per_arch or r['cv_f1_mean'] > best_per_arch[key]['cv_f1_mean']:
        best_per_arch[key] = r

summary = pd.DataFrame(best_per_arch.values())
summary['arch'] = summary['model'].apply(lambda x: 'MLP' if 'MLP' in x else x.split('-')[0] if '-' in x else x)

fig, ax = plt.subplots(figsize=(10, 5))
pivot_plot = summary.pivot_table(values='cv_f1_mean', index='model', columns='strategy', aggfunc='first')
if 'binary' in pivot_plot.columns and '3class' in pivot_plot.columns:
    pivot_plot = pivot_plot[['3class', 'binary']]
sns.heatmap(pivot_plot, annot=True, fmt='.4f', cmap='YlGn', ax=ax,
            linewidths=0.5, cbar_kws={'label': 'Val Macro F1'})
ax.set_title('DL Architecture Comparison')
ax.set_ylabel(''); ax.set_xlabel('')
plt.tight_layout()
plt.savefig('results/nb04a_dl_architectures.png', bbox_inches='tight')
plt.show()


In [ ]:
# Save
sb.to_csv('results/dl_architectures.csv', index=False)
print(f'\nSaved: results/dl_architectures.csv ({len(sb)} experiments)')
print(f'Best DL model per strategy:')
for strat in ['3class', 'binary']:
    sub = sb[sb['strategy'] == strat]
    if len(sub) > 0:
        best = sub.loc[sub['cv_f1_mean'].idxmax()]
        print(f'  {strat}: {best["model"]} (F1={best["cv_f1_mean"]:.4f})')
